# STRATA: teste completo de armazenamento num modelo grande

Este notebook faz, num modelo grande, o mesmo teste que já passou no Llama-3.1-8B (docs/05, seção 31):

1. baixa o **Q4_K_M publicado no Hub** (bartowski, com imatrix) e os **shards BF16 originais**, um shard por vez;
2. grava ao lado do GGUF os arquivos de estrato do STRATA: `e00` (normas), `e01`–`e03` (planos de 1 bit), `e04` (fecho até o BF16 exato) e o índice com SHA-256 de tudo. Cada shard BF16 é apagado depois de lido, para caber no disco;
3. **remonta todos os shards** a partir de GGUF + estratos e compara o SHA-256 de cada um com o do Hugging Face;
4. faz a conta de armazenamento com os tamanhos reais do Hub: BF16 + Q4_K_M + Q5_K_M + Q6_K hoje contra GGUF + estratos.

O que ele **não** mede: qualidade (perplexidade, KL). Os níveis intermediários continuam congelados e perdem para os k-quants com imatrix na mesma RAM; aqui o assunto é só armazenamento e volta exata.

## Ambiente

| | 70B (padrão) | 8B (reserva) |
|---|---|---|
| Modelo | `unsloth/Meta-Llama-3.1-70B-Instruct` (30 shards, 141,1 GB) | `unsloth/Meta-Llama-3.1-8B-Instruct` (4 shards, 16,1 GB) |
| GGUF | `bartowski/…-70B-Instruct-GGUF`, Q4_K_M 42,5 GB | `bartowski/…-8B-Instruct-GGUF`, Q4_K_M 4,9 GB |
| Disco livre | **≈ 115–120 GB** | ≈ 27–31 GB |
| RAM | ≥ 25 GB (High-RAM) | ≥ 12 GB |
| Tempo | ≈ 1–2 h com 8 núcleos (estimativa) | ≈ 15–20 min (medido: 559 s + 272 s em 4 núcleos) |

**Runtime recomendado:** *Ambiente de execução → Alterar o tipo* → CPU com **RAM alta** (ou TPU v6e‑1 com RAM alta, que costuma vir com mais disco). GPU não ajuda: o trabalho é de CPU e disco. A célula 2 mede o disco e escolhe sozinha: se não couber o 70B, roda o 8B e avisa.

Deixe a aba aberta durante a execução. No fim, a última célula baixa um `.zip` com os resultados; mande esse zip (ou o `resultado.json`) de volta na thread.

In [ ]:
# 1. Configuração
MODELO = "auto"     # "auto" (70B se couber, senão 8B), "70b" ou "8b"
NIVEIS = 3          # planos de 1 bit acima do Q4_K_M (o padrão dos testes)
RAMO = "claude/project-thread-btiyii"   # se o PR já foi mergeado e o ramo sumiu, a célula 3 usa o main
PASTA = "/content/strata_teste"

MODELOS = {
    "70b": dict(hf="unsloth/Meta-Llama-3.1-70B-Instruct",
                gguf_repo="bartowski/Meta-Llama-3.1-70B-Instruct-GGUF",
                gguf="Meta-Llama-3.1-70B-Instruct-Q4_K_M.gguf",
                gguf_gb=42.52, bf16_gb=141.11, shard_gb=5.00),
    "8b":  dict(hf="unsloth/Meta-Llama-3.1-8B-Instruct",
                gguf_repo="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF",
                gguf="Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf",
                gguf_gb=4.92, bf16_gb=16.06, shard_gb=4.98),
}

In [ ]:
# 2. A máquina: núcleos, RAM, disco, e a escolha do modelo
import os, shutil
os.makedirs(PASTA, exist_ok=True)
nuc = os.cpu_count()
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
livre = shutil.disk_usage(PASTA).free / 1e9
print(f"núcleos {nuc} · RAM {ram:.1f} GB · disco livre {livre:.1f} GB")

def precisa(m, adiante):
    # mesma conta do teste_grande.py: GGUF + resto (~40% do BF16) + shards à frente + folga
    return m["gguf_gb"] + 0.40 * m["bf16_gb"] + (adiante + 1) * m["shard_gb"] + 5

ESCOLHA, ADIANTE = None, 2
ordem = ["70b", "8b"] if MODELO == "auto" else [MODELO]
for k in ordem:
    for a in (2, 1):
        if livre - 2 >= precisa(MODELOS[k], a):  # 2 GB para o Rust e o repositório
            ESCOLHA, ADIANTE = k, a
            break
    if ESCOLHA:
        break
if not ESCOLHA:
    raise SystemExit(f"disco pequeno: o {ordem[-1]} precisa de ~{precisa(MODELOS[ordem[-1]], 1):.0f} GB livres")
if MODELO == "auto" and ESCOLHA != "70b":
    print(f"AVISO: o 70B precisa de ~{precisa(MODELOS['70b'], 1):.0f} GB livres; rodando o 8B.")
if ESCOLHA == "70b" and ram < 25:
    print("AVISO: pouca RAM para o 70B (o maior tensor tem ~1 G pesos); use o runtime com RAM alta.")
M = MODELOS[ESCOLHA]
print(f"modelo: {M['hf']} · disco estimado {precisa(M, ADIANTE):.0f} GB · {ADIANTE} shard(s) baixados à frente")

In [ ]:
# 3. Rust, o repositório e o binário strata-ref (≈ 3–5 min)
import subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, executable="/bin/bash")
    print(r.stdout[-3000:])
    if r.returncode:
        raise SystemExit(f"falhou: {cmd}")

if not os.path.exists(os.path.expanduser("~/.cargo/bin/cargo")):
    sh("curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
REPO = "/content/strata"
if not os.path.exists(REPO):
    r = subprocess.run(f"git clone -q --depth 1 -b {RAMO} https://github.com/cantyll/strata {REPO}", shell=True)
    if r.returncode:
        print(f"ramo {RAMO} não encontrado; usando o main")
        sh(f"git clone -q --depth 1 https://github.com/cantyll/strata {REPO}")
sh(f"cd {REPO} && git log -1 --oneline")
sh(f"cd {REPO}/leitor-rs && cargo build --release -q 2>&1 | tail -5; ls -la target/release/strata-ref")
sh("pip install -q -U huggingface_hub hf_xet psutil")

In [ ]:
# 4. Token do Hugging Face (opcional: só aumenta o limite de downloads; os repositórios são públicos)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN carregado dos Secrets do Colab")
except Exception:
    print("sem HF_TOKEN (tudo bem; para usar, crie o segredo HF_TOKEN no painel 🔑 à esquerda)")

In [ ]:
# 5. O teste: baixa, grava os estratos, remonta e confere o SHA-256 (a parte longa)
import sys, time
t0 = time.time()
cmd = [sys.executable, "-u", "tests/dist/teste_grande.py", "--hf", M["hf"], "--gguf-repo", M["gguf_repo"],
       "--gguf", M["gguf"], "--dir", PASTA, "--niveis", str(NIVEIS), "--adiante", str(ADIANTE)]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end="")
p.wait()
print(f"\n{'OK' if p.returncode == 0 else 'FALHOU (código %d)' % p.returncode} em {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 6. Relatório e download dos resultados
from IPython.display import Markdown, display
RES = os.path.join(PASTA, "resultados")
rel = os.path.join(RES, "relatorio.md")
if os.path.exists(rel):
    display(Markdown(open(rel).read()))
for f in ("model.strata.index.json",):  # o índice é pequeno e prova o que foi gravado
    src = os.path.join(PASTA, "dist", f)
    if os.path.exists(src):
        shutil.copy(src, RES)
with open(os.path.join(RES, "maquina.txt"), "w") as f:
    f.write(f"modelo {M['hf']}\nnúcleos {nuc}\nRAM {ram:.1f} GB\ndisco livre no início {livre:.1f} GB\nadiante {ADIANTE}\n")
zipado = shutil.make_archive(f"/content/strata_resultados_{ESCOLHA}", "zip", RES)
print(zipado, f"{os.path.getsize(zipado) / 1e6:.1f} MB")
try:
    from google.colab import files
    files.download(zipado)
except Exception:
    pass

## Como ler o resultado

- **SHA-256 igual ao do Hugging Face: N/N shards**: a volta é exata, bit a bit, a partir do GGUF publicado + estratos. Qualquer número menor que N é falha e precisa ser investigado antes de qualquer conclusão.
- **Total em bits/peso**: no 8B deu 11,13. Se o 70B ficar perto disso, a conta de armazenamento escala; se subir muito, o fecho (e04) é o primeiro suspeito.
- **Hub hoje → STRATA**: a economia vale para quem publica BF16 + Q4_K_M + Q5_K_M + Q6_K. Para quem só guarda o BF16, o formato 0.2 é ~5% menor.
- **Pico de RAM**: mostra se o escritor cabe numa máquina comum para esse tamanho.

Os arquivos de estrato (~55 GB no 70B) ficam só no disco do Colab e somem quando o runtime fecha; o zip leva o relatório, os logs e o índice.